# Part A — Warm-ups: Load & Look

In this section, the `houses.csv` dataset is loaded and explored using basic descriptive statistics and a scatter plot.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("houses.csv")

print(df.shape)
print(df.head())
print(df.describe())

### Part A Results

- The dataset contains **200 rows and 4 columns**.
- The mean house price is **89.911 ($1000s)**, approximately **$89,911**.
- The scatter plot shows a **positive relationship** between house size and price. In general, larger houses tend to have higher prices.

In [ ]:
plt.scatter(df["size"], df["price"])
plt.xlabel("size")
plt.ylabel("price")
plt.title("House Size vs Price")
plt.show()

# Part B — Simple Linear Regression

A simple linear regression model is fitted using `size` as the only feature.

The first 150 rows are used for training and the last 50 rows are reserved for testing.

In [ ]:
from sklearn.linear_model import LinearRegression

tr = df.iloc[:150]   # first 150 = training data
te = df.iloc[150:]   # last 50 = testing data

X = tr[["size"]]
y = tr["price"]

m = LinearRegression().fit(X, y)

print("slope    :", m.coef_[0])
print("intercept:", m.intercept_)

In [ ]:
prediction = m.predict(pd.DataFrame({"size": [18]}))[0]

print("predict size=18:", prediction)

### Part B Results

- **Slope:** 3.2851
- **Intercept:** 38.2297
- **Predicted price for size = 18:** 97.3619 ($1000s)

The slope of **3.2851** means that for every one-unit increase in house size, the predicted house price increases by approximately **$3,285**, while considering this simple model.

The intercept of **38.2297** represents the predicted price when the house size is zero. Although this is required mathematically to define the regression line, it has limited practical meaning because a zero-size house is not realistic.

For a house with size 18, the predicted price is approximately **$97,362**.

Python and Octave should produce approximately the same slope, intercept, and prediction when using the same data and training split.

# Part C — Evaluate Honestly

The size-only regression model is evaluated on the 50 test observations using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R².

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred = m.predict(te[["size"]])
yte = te["price"]

print("MAE :", mean_absolute_error(yte, pred))
print("RMSE:", np.sqrt(mean_squared_error(yte, pred)))
print("R2  :", r2_score(yte, pred))

### Part C Results

- **MAE:** 25.4793 ($1000s)
- **RMSE:** 31.4929 ($1000s)
- **R²:** 0.2397

The RMSE is relatively large compared with the mean house price of approximately $89,911. An RMSE of approximately $31,493 represents a substantial prediction error.

The R² value of approximately **0.24** means that the size-only model explains about **24% of the variation in house prices**. Therefore, house size alone is not sufficient to accurately predict house prices, and additional features are likely needed.

# Part D — Multiple Linear Regression

The model is extended to use three features: `size`, `rooms`, and `age`.

This allows the model to consider multiple factors that may influence house price.

In [ ]:
feats = ["size", "rooms", "age"]

m3 = LinearRegression().fit(tr[feats], tr["price"])

print("coefficients:", m3.coef_)
print("intercept   :", m3.intercept_)

In [ ]:
pred3 = m3.predict(te[feats])

print("test R2  :", r2_score(te["price"], pred3))
print("test RMSE:", np.sqrt(mean_squared_error(te["price"], pred3)))

### Part D Results

The fitted model has the following coefficients:

- **Size:** 3.0913
- **Rooms:** 14.4436
- **Age:** -1.3835
- **Intercept:** 26.5102

The regression equation is approximately:

**price = 26.5102 + 3.0913(size) + 14.4436(rooms) - 1.3835(age)**

The size coefficient indicates that increasing house size by one unit is associated with an approximately **$3,091** increase in predicted price, while holding the other features constant.

The rooms coefficient indicates that adding one room is associated with an approximately **$14,444** increase in predicted price, while holding the other features constant.

The age coefficient is negative, meaning that each additional year of house age is associated with an approximately **$1,384 decrease** in predicted price, while holding the other features constant.

The model achieved:

- **Test R²:** 0.9073
- **Test RMSE:** 10.9956 ($1000s)

Compared with the size-only model from Part C, the multiple regression model performs significantly better. R² increased from **0.2397 to 0.9073**, while RMSE decreased from **31.4929 to 10.9956**.

Therefore, adding `rooms` and `age` provides substantial predictive value.

Python and Octave should produce approximately the same results because both use the same training/test split and linear regression formulation.

# Part E — Features vs Complexity

This section investigates how adding relevant and irrelevant features affects model performance.

The test R² is compared across models with different feature combinations.

## E1 — Comparing Feature Combinations

Three models are evaluated:

1. `size`
2. `size + rooms`
3. `size + rooms + age`

In [ ]:
models = [
    ["size"],
    ["size", "rooms"],
    ["size", "rooms", "age"]
]

for feats in models:
    model = LinearRegression().fit(tr[feats], tr["price"])
    pred = model.predict(te[feats])
    r2 = r2_score(te["price"], pred)

    print(feats, "-> Test R2:", round(r2, 3))

### E1 Results

| Features | Test R² |
|---|---:|
| Size | 0.240 |
| Size + Rooms | 0.749 |
| Size + Rooms + Age | 0.907 |

As relevant features are added, the test R² increases substantially.

The size-only model explains approximately 24% of the variation in price. Adding `rooms` increases this to approximately 75%, while adding `age` further increases it to approximately 91%.

This shows that both `rooms` and `age` provide useful information for predicting house prices.

## E2 — Adding a Useless Feature

A random `noise` feature is added to the dataset to investigate whether simply increasing the number of features improves model performance.

In [ ]:
tr = tr.copy()
te = te.copy()

tr["noise"] = np.random.rand(len(tr))
te["noise"] = np.random.rand(len(te))

In [ ]:
m4 = LinearRegression().fit(
    tr[["size", "rooms", "age", "noise"]],
    tr["price"]
)

pred4 = m4.predict(
    te[["size", "rooms", "age", "noise"]]
)

print(
    "with noise -> Test R2:",
    r2_score(te["price"], pred4)
)

### E2 Results

The original model achieved a test R² of **0.9073**.

After adding the random noise feature, the test R² was approximately **0.9052**.

The test R² did not improve. Instead, it decreased slightly.

This demonstrates that adding more features does not automatically improve a model. Irrelevant features can add noise and increase model complexity without providing useful predictive information. This is related to the risk of **overfitting**.